# 42 -- Steps 2-6 under the new encoding v = a e+ + b e-

New encoding decision for section 01e: the visible input is
`v = a*e+ + b*e-`, so channel + receives `a` (this movie's normalized
rating) and channel - receives `b` (this user's normalized mean rating),
for every movie in the rated movie list; unrated movies stay 0 in both
channels. This replaces `v = a + b*k` and the old channel inputs
`a+b` / `a-b` for Steps 2-6.

Reuses notebook 37's exact data loading, user 24127, Rumble in the Bronx,
h1, starting weights, biases, learning rate and random-draw procedure.
Only the channel inputs change. Does not touch the HTML.

## Part 0 -- Setup: scan notebooks/outputs, pick this notebook's number

In [1]:
from pathlib import Path
import json
import re
import math

import pandas as pd
import torch

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
notebooks_dir = root / "notebooks"
data_dir = root / "data"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

nb_paths = sorted(p for p in notebooks_dir.glob("*.ipynb") if "e_plus_e_minus_encoding" not in p.stem)
num_re = re.compile(r"^(\d+)")
numbered = sorted(
    (int(m.group(1)), p) for p in nb_paths if (m := num_re.match(p.stem))
)
highest_num, highest_path = numbered[-1]
this_num = highest_num + 1
this_name = f"{this_num}_step2to6_e_plus_e_minus_encoding.ipynb"
print(f"highest existing notebook: {highest_num:>3d}  ({highest_path.name})")
print(f"this notebook is:          {this_num:>3d}  ({this_name})")

highest existing notebook:  41  (41_step2_rewrite_verification.ipynb)
this notebook is:           42  (42_step2to6_e_plus_e_minus_encoding.ipynb)


## Part 1 -- Reuse notebook 37's exact setup

In [2]:
train_path = data_dir / "train_hyperbolic.csv"
test_path = data_dir / "test_hyperbolic.csv"
results31_path = out_dir / "31_step8_results.json"
results37_path = out_dir / "37_step1to6_rating_habit_b_results.json"
for p in (train_path, test_path, results31_path, results37_path):
    assert p.exists(), f"Missing {p}"

train_hyp = pd.read_csv(train_path)
test_hyp = pd.read_csv(test_path)
assert len(train_hyp) == 492055 and len(test_hyp) == 123474

results31 = json.loads(results31_path.read_text())
results37 = json.loads(results37_path.read_text())
REAL_SEED = results31["seeds"]["plus_seed_reused"]
MINUS_SEED = results31["seeds"]["minus_seed_trained"]
N_HIDDEN = int(results31["architecture_match"]["128 hidden units"].split("=")[1])
LR = results31["worked_example_plus"]["lr"]
H1_IDX = 0

all_users = sorted(pd.concat([train_hyp["userId"], test_hyp["userId"]]).unique())
all_movies = sorted(pd.concat([train_hyp["movieId"], test_hyp["movieId"]]).unique())
n_movies = len(all_movies)
movie_to_col = {m: j for j, m in enumerate(all_movies)}

movies = pd.read_csv(data_dir / "movie.csv")
RUMBLE_ID = int(movies[movies["title"].str.contains("Rumble in the Bronx", case=False, na=False)].iloc[0]["movieId"])
RUMBLE_TITLE = movies[movies["movieId"] == RUMBLE_ID].iloc[0]["title"]
rumble_col = movie_to_col[RUMBLE_ID]
MEDIAN_USER = results31["worked_example_plus"]["user"]
assert MEDIAN_USER == 24127

user_rows = train_hyp[train_hyp["userId"] == MEDIAN_USER]
v0_full = torch.zeros(n_movies)  # a_i per rated movie -- used for the 01c baseline (both channels)
mask0_full = torch.zeros(n_movies)
for row in user_rows.itertuples():
    v0_full[movie_to_col[row.movieId]] = row.a
    mask0_full[movie_to_col[row.movieId]] = 1.0
rated_idx = mask0_full.nonzero(as_tuple=True)[0]
a_rumble = v0_full[rumble_col].item()
n_rated = len(rated_idx)
assert n_rated == 530
mask0b = mask0_full.unsqueeze(0)

g_init_p = torch.Generator().manual_seed(REAL_SEED)
W_plus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_p) * 0.01
hbias_plus_init = torch.zeros(N_HIDDEN)
vbias_plus_init = torch.zeros(n_movies)

g_init_m = torch.Generator().manual_seed(MINUS_SEED)
W_minus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_m) * 0.01
hbias_minus_init = torch.zeros(N_HIDDEN)
vbias_minus_init = torch.zeros(n_movies)

b_val = results37["step1"]["b"]
w_plus_rumble = W_plus_init[rumble_col, H1_IDX].item()
w_minus_rumble = W_minus_init[rumble_col, H1_IDX].item()
print(f"a[Rumble]={a_rumble:.6f}  b={b_val:.6f}  n_rated={n_rated}")
print(f"w+[Rumble,h1]={w_plus_rumble:.6f}  w-[Rumble,h1]={w_minus_rumble:.6f}")

a[Rumble]=0.777778  b=0.656813  n_rated=530
w+[Rumble,h1]=0.006433  w-[Rumble,h1]=-0.009362


## Part 2 -- New channel inputs: v+ = a (per movie), v- = b (constant over R)

In [3]:
v_plus_new = v0_full.clone()  # channel + still receives a_i per rated movie -- identical to the 01c baseline
v_minus_new = torch.where(mask0_full.bool(), torch.full((n_movies,), b_val), torch.zeros(n_movies))  # channel - = b, constant over R

print(f"v_plus_new[Rumble] = {v_plus_new[rumble_col].item():.6f}  (= a, unchanged from 01c)")
print(f"v_minus_new[Rumble] = {v_minus_new[rumble_col].item():.6f}  (= b, same for every rated movie)")
print(f"v_minus_new min/max over rated set = {v_minus_new[rated_idx].min().item():.6f} / {v_minus_new[rated_idx].max().item():.6f} (constant, as expected)")

v_plus_new[Rumble] = 0.777778  (= a, unchanged from 01c)
v_minus_new[Rumble] = 0.656813  (= b, same for every rated movie)
v_minus_new min/max over rated set = 0.656813 / 0.656813 (constant, as expected)


## Part 3 -- Step 2: Rumble term, other 529 movies, 01c part, sum over R, b term (replacement), x+-

Channel + is unchanged from 01c (still `a_i` per movie), so its x+ is
exactly the 01c value; there is no separate b term to add. Channel - no
longer varies by movie -- it is `b` times the weight, summed over R --
so its own "b term" is folded directly into the 01c-part sum; the
replacement b term is 0 for both channels.

In [4]:
x_plus_01c = (v0_full @ W_plus_init + hbias_plus_init)[H1_IDX].item()
x_minus_01c = (v0_full @ W_minus_init + hbias_minus_init)[H1_IDX].item()
assert abs(x_plus_01c - results37["step2"]["x_plus"]["01c"]) < 1e-6
assert abs(x_minus_01c - results37["step2"]["x_minus"]["01c"]) < 1e-6

x_plus_new = (v_plus_new @ W_plus_init + hbias_plus_init)[H1_IDX].item()
x_minus_new = (v_minus_new @ W_minus_init + hbias_minus_init)[H1_IDX].item()
assert abs(x_plus_new - x_plus_01c) < 1e-6, "channel + is unchanged from 01c under the new encoding"

rumble_term_plus_new = v_plus_new[rumble_col].item() * w_plus_rumble
rumble_term_minus_new = v_minus_new[rumble_col].item() * w_minus_rumble
other_idx = rated_idx[rated_idx != rumble_col]
assert len(other_idx) == 529
other_plus_new = (v_plus_new[other_idx] @ W_plus_init[other_idx, H1_IDX]).item()
other_minus_new = (v_minus_new[other_idx] @ W_minus_init[other_idx, H1_IDX]).item()
part_01c_plus_new = rumble_term_plus_new + other_plus_new + hbias_plus_init[H1_IDX].item()
part_01c_minus_new = rumble_term_minus_new + other_minus_new + hbias_minus_init[H1_IDX].item()
assert abs(part_01c_plus_new - x_plus_new) < 1e-6
assert abs(part_01c_minus_new - x_minus_new) < 1e-6

b_term_plus_new = 0.0
b_term_minus_new = 0.0

sum_w_plus_R = W_plus_init[rated_idx, H1_IDX].sum().item()
sum_w_minus_R = W_minus_init[rated_idx, H1_IDX].sum().item()

print(f"Rumble term:   +={rumble_term_plus_new:.6f}  -={rumble_term_minus_new:.6f}")
print(f"other 529:     +={other_plus_new:.6f}  -={other_minus_new:.6f}")
print(f"01c part:      +={part_01c_plus_new:.6f}  -={part_01c_minus_new:.6f}")
print(f"sum_R W:       +={sum_w_plus_R:.6f}  -={sum_w_minus_R:.6f}")
print(f"b term (repl): +={b_term_plus_new}  -={b_term_minus_new}")
print(f"x (new):       +={x_plus_new:.6f}  -={x_minus_new:.6f}")
print(f"x (01c):       +={x_plus_01c:.6f}  -={x_minus_01c:.6f}")

Rumble term:   +=0.005004  -=-0.006149
other 529:     +=-0.216119  -=-0.207563
01c part:      +=-0.211115  -=-0.213712
sum_R W:       +=-0.332340  -=-0.325376
b term (repl): +=0.0  -=0.0
x (new):       +=-0.211115  -=-0.213712
x (01c):       +=-0.211115  -=-0.205367


## Part 4 -- Step 3: q+, q-, q_r, q_k (new encoding and 01c side by side)

In [5]:
def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))

q_plus_new, q_minus_new = sigmoid(x_plus_new), sigmoid(x_minus_new)
q_plus_01c, q_minus_01c = sigmoid(x_plus_01c), sigmoid(x_minus_01c)
q_r_new = (q_plus_new + q_minus_new) / 2
q_k_new = (q_plus_new - q_minus_new) / 2
q_r_01c = (q_plus_01c + q_minus_01c) / 2
q_k_01c = (q_plus_01c - q_minus_01c) / 2

print(f"q+ : new={q_plus_new:.6f}  01c={q_plus_01c:.6f}")
print(f"q- : new={q_minus_new:.6f}  01c={q_minus_01c:.6f}")
print(f"q_r: new={q_r_new:.6f}  01c={q_r_01c:.6f}")
print(f"q_k: new={q_k_new:.6f}  01c={q_k_01c:.6f}")

q+ : new=0.447416  01c=0.447416
q- : new=0.446775  01c=0.448838
q_r: new=0.447095  01c=0.448127
q_k: new=0.000321  01c=-0.000711


## Part 5 -- Step 4: joint state probabilities (new encoding)

In [6]:
P_00_new = (1 - q_plus_new) * (1 - q_minus_new)
P_10_new = q_plus_new * (1 - q_minus_new)
P_01_new = (1 - q_plus_new) * q_minus_new
P_11_new = q_plus_new * q_minus_new
assert abs(P_00_new + P_10_new + P_01_new + P_11_new - 1.0) < 1e-9

P_00_01c = (1 - q_plus_01c) * (1 - q_minus_01c)
P_10_01c = q_plus_01c * (1 - q_minus_01c)
P_01_01c = (1 - q_plus_01c) * q_minus_01c
P_11_01c = q_plus_01c * q_minus_01c

print(f"P(0,0): new={P_00_new:.6f}  01c={P_00_01c:.6f}")
print(f"P(1,0): new={P_10_new:.6f}  01c={P_10_01c:.6f}")
print(f"P(0,1): new={P_01_new:.6f}  01c={P_01_01c:.6f}")
print(f"P(1,1): new={P_11_new:.6f}  01c={P_11_01c:.6f}")

P(0,0): new=0.305703  01c=0.304563
P(1,0): new=0.247522  01c=0.246599
P(0,1): new=0.246880  01c=0.248020
P(1,1): new=0.199894  01c=0.200817


## Part 6 -- Step 5: full 128-unit sampling and reconstruction (new encoding and 01c)

Under this encoding, channel + reconstructs a rating directly
(`r_hat = 4.5*v_hat+ + 0.5`) and channel - reconstructs the mean signal
directly (`b_hat = v_hat-`), rather than averaging/differencing the two
channels as in the hyperbolic (v = a + bk) framing.

In [7]:
def cd1_round_trip(v_vec, W, hbias, vbias, mask, seed):
    vb = v_vec.unsqueeze(0)
    h0_prob = torch.sigmoid(vb @ W + hbias)
    g = torch.Generator().manual_seed(seed)
    h0_sample = torch.bernoulli(h0_prob, generator=g)
    v1_prob = torch.sigmoid(h0_sample @ W.t() + vbias) * mask
    h1_prob = torch.sigmoid(v1_prob @ W + hbias)
    return h0_prob, h0_sample, v1_prob, h1_prob

h0_plus_new, s_plus_new, v1_plus_new, h1_plus_new = cd1_round_trip(v_plus_new, W_plus_init, hbias_plus_init, vbias_plus_init, mask0b, REAL_SEED)
h0_minus_new, s_minus_new, v1_minus_new, h1_minus_new = cd1_round_trip(v_minus_new, W_minus_init, hbias_minus_init, vbias_minus_init, mask0b, MINUS_SEED)
h0_plus_01c, s_plus_01c, v1_plus_01c, h1_plus_01c = cd1_round_trip(v0_full, W_plus_init, hbias_plus_init, vbias_plus_init, mask0b, REAL_SEED)
h0_minus_01c, s_minus_01c, v1_minus_01c, h1_minus_01c = cd1_round_trip(v0_full, W_minus_init, hbias_minus_init, vbias_minus_init, mask0b, MINUS_SEED)

vhat_plus_new = v1_plus_new[0, rumble_col].item()
vhat_minus_new = v1_minus_new[0, rumble_col].item()
vhat_plus_01c = v1_plus_01c[0, rumble_col].item()
vhat_minus_01c = v1_minus_01c[0, rumble_col].item()
assert abs(vhat_plus_01c - results37["step5"]["vhat_plus"]["01c"]) < 1e-4
assert abs(vhat_minus_01c - results37["step5"]["vhat_minus"]["01c"]) < 1e-4

rhat_new = 4.5 * vhat_plus_new + 0.5
bhat_new = vhat_minus_new
rhat_01c = 4.5 * vhat_plus_01c + 0.5
bhat_01c = vhat_minus_01c

print(f"v_hat+ : new={vhat_plus_new:.6f}  01c={vhat_plus_01c:.6f}")
print(f"v_hat- : new={vhat_minus_new:.6f}  01c={vhat_minus_01c:.6f}")
print(f"r_hat (=4.5*v_hat+ +0.5): new={rhat_new:.6f}  01c={rhat_01c:.6f}")
print(f"b_hat (=v_hat-):          new={bhat_new:.6f}  01c={bhat_01c:.6f}")

v_hat+ : new=0.488463  01c=0.488463
v_hat- : new=0.460132  01c=0.463210
r_hat (=4.5*v_hat+ +0.5): new=2.698085  01c=2.698085
b_hat (=v_hat-):          new=0.460132  01c=0.463210


## Part 7 -- Step 6: CD-1 update, dW+-, dw_r, dw_k (new encoding and 01c)

In [8]:
q0_plus_new = h0_plus_new[0, H1_IDX].item()
q0_minus_new = h0_minus_new[0, H1_IDX].item()
q1_plus_new = h1_plus_new[0, H1_IDX].item()
q1_minus_new = h1_minus_new[0, H1_IDX].item()
q0_plus_01c = h0_plus_01c[0, H1_IDX].item()
q0_minus_01c = h0_minus_01c[0, H1_IDX].item()
q1_plus_01c = h1_plus_01c[0, H1_IDX].item()
q1_minus_01c = h1_minus_01c[0, H1_IDX].item()

v0_plus_rumble_new = v_plus_new[rumble_col].item()
v0_minus_rumble_new = v_minus_new[rumble_col].item()

dW_plus_new = LR * (v0_plus_rumble_new * q0_plus_new - vhat_plus_new * q1_plus_new)
dW_minus_new = LR * (v0_minus_rumble_new * q0_minus_new - vhat_minus_new * q1_minus_new)
dW_plus_01c = LR * (a_rumble * q0_plus_01c - vhat_plus_01c * q1_plus_01c)
dW_minus_01c = LR * (a_rumble * q0_minus_01c - vhat_minus_01c * q1_minus_01c)
assert abs(dW_plus_01c - 0.0062) < 5e-4
assert abs(dW_minus_01c - 0.0067) < 5e-4

dw_r_new = (dW_plus_new + dW_minus_new) / 2
dw_k_new = (dW_plus_new - dW_minus_new) / 2
dw_r_01c = (dW_plus_01c + dW_minus_01c) / 2
dw_k_01c = (dW_plus_01c - dW_minus_01c) / 2

print(f"dW+  : new={dW_plus_new:.6f}  01c={dW_plus_01c:.6f}")
print(f"dW-  : new={dW_minus_new:.6f}  01c={dW_minus_01c:.6f}")
print(f"dw_r : new={dw_r_new:.6f}  01c={dw_r_01c:.6f}")
print(f"dw_k : new={dw_k_new:.6f}  01c={dw_k_01c:.6f}")

dW+  : new=0.006163  01c=0.006163
dW-  : new=0.003967  01c=0.006682
dw_r : new=0.005065  01c=0.006422
dw_k : new=0.001098  01c=-0.000259


## Part 8 -- Save results and print the full table

In [9]:
results = {
    "notebook": this_name,
    "reused_from": "37_step1to6_rating_habit_b.ipynb",
    "encoding": "v = a*e+ + b*e-; channel+ = a per rated movie, channel- = b (constant) per rated movie, 0 unrated",
    "movie": RUMBLE_TITLE,
    "user": MEDIAN_USER,
    "hidden_unit": "h1 (index 0)",
    "a": a_rumble,
    "b": b_val,
    "n_rated": n_rated,
    "step2": {
        "rumble_term_plus": {"new": rumble_term_plus_new}, "rumble_term_minus": {"new": rumble_term_minus_new},
        "other_529_plus": {"new": other_plus_new}, "other_529_minus": {"new": other_minus_new},
        "part_01c_plus": {"new": part_01c_plus_new}, "part_01c_minus": {"new": part_01c_minus_new},
        "sum_w_plus_R": sum_w_plus_R, "sum_w_minus_R": sum_w_minus_R,
        "b_term_plus": {"new": b_term_plus_new}, "b_term_minus": {"new": b_term_minus_new},
        "x_plus": {"new": x_plus_new, "01c": x_plus_01c},
        "x_minus": {"new": x_minus_new, "01c": x_minus_01c},
    },
    "step3": {
        "q_plus": {"new": q_plus_new, "01c": q_plus_01c},
        "q_minus": {"new": q_minus_new, "01c": q_minus_01c},
        "q_r": {"new": q_r_new, "01c": q_r_01c},
        "q_k": {"new": q_k_new, "01c": q_k_01c},
    },
    "step4": {
        "P_00": {"new": P_00_new, "01c": P_00_01c},
        "P_10": {"new": P_10_new, "01c": P_10_01c},
        "P_01": {"new": P_01_new, "01c": P_01_01c},
        "P_11": {"new": P_11_new, "01c": P_11_01c},
    },
    "step5": {
        "vhat_plus": {"new": vhat_plus_new, "01c": vhat_plus_01c},
        "vhat_minus": {"new": vhat_minus_new, "01c": vhat_minus_01c},
        "rhat": {"new": rhat_new, "01c": rhat_01c},
        "bhat": {"new": bhat_new, "01c": bhat_01c},
    },
    "step6": {
        "dW_plus": {"new": dW_plus_new, "01c": dW_plus_01c},
        "dW_minus": {"new": dW_minus_new, "01c": dW_minus_01c},
        "dw_r": {"new": dw_r_new, "01c": dw_r_01c},
        "dw_k": {"new": dw_k_new, "01c": dw_k_01c},
    },
}
results_path = out_dir / "42_step2to6_e_plus_e_minus_encoding_results.json"
with results_path.open("w") as f:
    json.dump(results, f, indent=2)
print(f"saved {results_path}\n")

print(f"{'quantity':<16}{'new encoding':>16}{'01c':>16}")
rows = [
    ("Rumble term +", rumble_term_plus_new, None), ("Rumble term -", rumble_term_minus_new, None),
    ("other529 +", other_plus_new, None), ("other529 -", other_minus_new, None),
    ("01c part +", part_01c_plus_new, None), ("01c part -", part_01c_minus_new, None),
    ("sum_R W+", sum_w_plus_R, None), ("sum_R W-", sum_w_minus_R, None),
    ("b term +", b_term_plus_new, None), ("b term -", b_term_minus_new, None),
    ("x+", x_plus_new, x_plus_01c), ("x-", x_minus_new, x_minus_01c),
    ("q+", q_plus_new, q_plus_01c), ("q-", q_minus_new, q_minus_01c),
    ("q_r", q_r_new, q_r_01c), ("q_k", q_k_new, q_k_01c),
    ("P(0,0)", P_00_new, P_00_01c), ("P(1,0)", P_10_new, P_10_01c),
    ("P(0,1)", P_01_new, P_01_01c), ("P(1,1)", P_11_new, P_11_01c),
    ("v_hat+", vhat_plus_new, vhat_plus_01c), ("v_hat-", vhat_minus_new, vhat_minus_01c),
    ("r_hat", rhat_new, rhat_01c), ("b_hat", bhat_new, bhat_01c),
    ("dW+", dW_plus_new, dW_plus_01c), ("dW-", dW_minus_new, dW_minus_01c),
    ("dw_r", dw_r_new, dw_r_01c), ("dw_k", dw_k_new, dw_k_01c),
]
for name, new_val, old_val in rows:
    old_str = f"{old_val:.6f}" if old_val is not None else "--"
    print(f"{name:<16}{new_val:>16.6f}{old_str:>16}")

saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/outputs/42_step2to6_e_plus_e_minus_encoding_results.json

quantity            new encoding             01c
Rumble term +           0.005004              --
Rumble term -          -0.006149              --
other529 +             -0.216119              --
other529 -             -0.207563              --
01c part +             -0.211115              --
01c part -             -0.213712              --
sum_R W+               -0.332340              --
sum_R W-               -0.325376              --
b term +                0.000000              --
b term -                0.000000              --
x+                     -0.211115       -0.211115
x-                     -0.213712       -0.205367
q+                      0.447416        0.447416
q-                      0.446775        0.448838
q_r                     0.447095        0.448127
q_k                     0.000321       -0.000711
P(0,0)                  0.305703        0.3045